Language: **English (default)** | [Polski](lab.ipynb)

# Lab 11 -- designing and evaluating a small LLM inference pipeline

Companion material to [`README.md`](README.md). Pipeline: prompt ->
tokenization -> model -> generation -> decoding -> measurement ->
evaluation.

**Mandatory core: sections 0-11 (about 105 min + 15 min buffer = 120
min).** Sections R1-R2 at the end of the notebook are **optional
extensions**.

In [1]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"

/Users/michalmaj/Documents/python/deep-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 1. Model and tokenizer (provided + short task)

In [2]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"load time: {time.time() - t0:.2f}s")
print(f"number of parameters: {sum(p.numel() for p in model.parameters()):,}")

W1019 14:21:33.701000 87283 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:52,  5.16it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 959.42it/s]

load time: 1.10s
number of parameters: 134,515,008


In [3]:
# TODO: print model.config.num_attention_heads and model.config.num_key_value_heads
# Are they equal? What does this mean for KV cache memory cost? (see the lecture)
raise NotImplementedError("TODO: check Q vs KV heads")

NotImplementedError: TODO: check Q vs KV heads

## 2. Preparing prompts -- your task

Choose **exactly 3 required** prompts, covering:
1. **one** unambiguous "information extraction" prompt (has a precise
   expected answer),
2. **a second** unambiguous prompt, of a different type (e.g. "format
   transformation"),
3. **one qualitative** prompt (a paraphrase or continuation --
   `expected=None`).

For **every** prompt with an unambiguous answer, write that answer down
**in advance**, before you see what the model generates. Structure: a
list of dicts `{"id", "type", "prompt", "expected"}`.

**Extension R1 (optional):** add 2 more prompts (e.g. a question with
context, a continuation) with full treatment as in the core.

In [4]:
# TODO: build a list of EXACTLY 3 required dicts {"id","type","prompt","expected"}:
# 1. one unambiguous prompt (e.g. information extraction), expected != None
# 2. a SECOND unambiguous prompt, of a different type (e.g. format transformation)
# 3. one QUALITATIVE prompt (a paraphrase or continuation), expected=None
prompts = []
raise NotImplementedError("TODO: preparing prompts")

NotImplementedError: TODO: preparing prompts

## 3. Chat template -- your task

Implement `format_prompt(prompt_text)`, which returns `inputs` ready for
`model.generate(**inputs, ...)`, using `tokenizer.apply_chat_template()`
with `add_generation_prompt=True`. **Do not** manually add any special
tokens -- `apply_chat_template` already inserts them (see the lecture,
"Chat templates" section).

**API note (verified in the lecture):** `apply_chat_template(...,
tokenize=True, return_tensors="pt")` **without** `return_dict=True` may
return a different type than you expect in your `transformers` version --
use `return_dict=True` explicitly and access it via `inputs["input_ids"]`,
do not assume a bare tensor.

**Task (code):** verify that your function actually adds the special
tokens (check that the decoded prompt starts with the BOS token/
`<|im_start|>`, not with the prompt's content).

In [5]:
def format_prompt(prompt_text):
    # TODO:
    # 1. build messages = [{"role": "user", "content": prompt_text}]
    # 2. return tokenizer.apply_chat_template(messages, tokenize=True,
    #    add_generation_prompt=True, return_tensors="pt", return_dict=True)
    raise NotImplementedError("TODO: format_prompt")

**Verification (a provided assertion): the decoded start of the
formatted prompt must begin with a special token (BOS/`<|im_start|>`),
not with the content.**

In [6]:
check_inputs = format_prompt("test")  # noqa: F821 - defined in section 3
start_decoded = tokenizer.decode(check_inputs["input_ids"][0][:1])
print(f"first token of the formatted prompt: {start_decoded!r}")
assert start_decoded == tokenizer.bos_token, (
    f"BUG: expected {tokenizer.bos_token!r} as the first token, got {start_decoded!r}"
)
print("OK: format_prompt adds a special token at the start.")

NotImplementedError: TODO: format_prompt

## 4. Generation and correctly decoding the result -- your task

Implement `generate_response(prompt_text, **gen_kwargs)`:
1. format the prompt (section 3),
2. call `model.generate(**inputs, **gen_kwargs, pad_token_id=...)`,
3. **extract ONLY the newly generated tokens** (not the whole `output`,
   which also contains the prompt -- see the lecture, Experiment B/E),
4. decode (`skip_special_tokens=True`),
5. return the text **and** the number of newly generated tokens (needed
   for the throughput measurement in section 6).

In [7]:
def generate_response(prompt_text, **gen_kwargs):
    # TODO:
    # 1. inputs = format_prompt(prompt_text)
    # 2. prompt_len = inputs["input_ids"].shape[1]
    # 3. out = model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    # 4. EXTRACT ONLY the newly generated tokens: out[0, prompt_len:] (NOT all of out!)
    # 5. decode (skip_special_tokens=True)
    # 6. return (text, number_of_new_tokens)
    raise NotImplementedError("TODO: generate_response")

**Verification (a provided assertion): for `do_sample=False`, two calls
with the same prompt must give an identical result.**

In [8]:
r1, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
r2, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
print(f"run 1: {r1!r}")
print(f"run 2: {r2!r}")
assert r1 == r2, "BUG: greedy decoding should be deterministic"
print("OK: deterministic repeats are identical.")

NotImplementedError: TODO: generate_response

## 5. Deterministic generation and sampling -- your task

For **all 3 required** prompts (section 2), generate a **greedy**
(`do_sample=False`) response. Additionally, for a **subset** (the
unambiguous prompt `extract` **and** the qualitative prompt `paraphrase`
-- **not** `format`, to avoid requiring the full cross-product of 3
prompts x 3 strategies), also run **sampling** with two temperatures
(0.7 and 1.3). Use a **fixed seed** (`torch.manual_seed(...)`) before
every sampling call.

**Extension R2 (optional):** also run sampling for `format` (and for the
R1 prompts, if you added them) -- does sampling help on unambiguous
tasks the core does not give it to?

In [9]:
configs_greedy = [("greedy", dict(do_sample=False))]
configs_sampling = [
    ("sampling tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling tau=1.3", dict(do_sample=True, temperature=1.3)),
]
SAMPLING_SUBSET = {"extract", "paraphrase"}  # prompt ids that additionally get sampling

# TODO:
# for each prompt (section 2): always greedy; ADDITIONALLY sampling (both variants
# from configs_sampling) ONLY if p["id"] in SAMPLING_SUBSET
# 1. set torch.manual_seed(0) BEFORE every call (reproducibility)
# 2. measure the time (time.time() before/after) of the call
#    generate_response(prompt, max_new_tokens=40, **kwargs)
# 3. collect the results (a list of dicts: id, strategy, text, n_new, time, tok_s)
results = []
raise NotImplementedError("TODO: generation - greedy for all, sampling for the subset")

NotImplementedError: TODO: generation - greedy for all, sampling for the subset

## 6. Measuring time and throughput (provided -- already collected in section 5)

**Task:** print a tokens/second table for every (prompt, strategy)
combination in `results`.

In [10]:
# TODO: print id, strategy, and tok/s for every entry in `results`
raise NotImplementedError("TODO: throughput table")

NotImplementedError: TODO: throughput table

## 7. Effect of prompt length -- your task

Choose **one** prompt and generate a longer version of it (e.g. add
context/history). Measure the generation time for **the same number**
of new tokens for the short and long versions of the prompt.

**Important:** use `min_new_tokens == max_new_tokens` to force **the
same** number of new tokens for both prompt lengths -- otherwise
differences in generation time (e.g. stopping early via EOS) would
contaminate the measurement of the effect of prompt length alone (this
pitfall is real -- confirmed while preparing this lab).

In [11]:
short_prompt = "What is 2+2?"
long_prompt = ("Here is a lot of context to pad out the prompt. " * 30) + " What is 2+2?"

# TODO: for both prompts, measure the time of generate_response with max_new_tokens=20 AND
# min_new_tokens=20 (the same number of new tokens for both!), print prompt_len and time
raise NotImplementedError("TODO: effect of prompt length")

NotImplementedError: TODO: effect of prompt length

## 8. KV cache -- your task

In [12]:
# TODO: for use_cache=True and use_cache=False, measure the time of generate_response
# ("Tell me about the solar system.", max_new_tokens=40, do_sample=False, use_cache=...)
raise NotImplementedError("TODO: KV cache")

NotImplementedError: TODO: KV cache

## 9. Evaluating quality -- your task

1. **Automatically**, for **both** prompts with an unambiguous answer
   (section 2) -- on their **greedy** response -- check
   `p["expected"].lower() in response.lower()`.
2. **Qualitatively**, for the **qualitative** prompt (section 2) --
   evaluate **every** strategy run (greedy + 2 sampling variants from
   section 5) against the five criteria from the README (correct
   execution, completeness, format compliance, naturalness,
   reproducibility) -- with a **specific justification based on a
   fragment of the answer**, not "looks good."

❌ "This answer looks better, so it is better."

✅ We separate these criteria precisely so we do **not** evaluate with
one, subjective impression.

**Extension R1/R2 (optional):** if you did R1 (extra prompts) or R2
(extra sampling), evaluate them with the same scheme.

In [13]:
# TODO: for both prompts with expected != None, check whether
# p["expected"].lower() in <that prompt's greedy response>.lower()
# print id, expected, the check result, and the response itself
raise NotImplementedError("TODO: automatic evaluation")

NotImplementedError: TODO: automatic evaluation

**Task (markdown):** for the `paraphrase` prompt (the only qualitative
prompt in the core), evaluate **every** strategy you ran (greedy,
tau=0.7, tau=1.3) for: whether it preserves the original's meaning,
whether it is complete, whether it sounds natural. A specific
justification, not "looks good."

_(Your answer here.)_

## 10. Diagnostic task

The following code **looks** correct -- it raises no exception, it
generates text -- but it has a bug: confusing `max_length` with
`max_new_tokens`.

In [14]:
DESIRED_NEW_TOKENS = 40
diag_prompt = "Write a short poem about the ocean."
diag_inputs = format_prompt(diag_prompt)  # noqa: F821
diag_prompt_len = diag_inputs["input_ids"].shape[1]
print(f"prompt_len={diag_prompt_len}, we want {DESIRED_NEW_TOKENS} NEW tokens")

with torch.no_grad():
    out_buggy = model.generate(
        **diag_inputs, max_length=DESIRED_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
actual_new_buggy = out_buggy.shape[1] - diag_prompt_len
print(f"BuggyGeneration: actual new tokens = {actual_new_buggy} "
      f"(we wanted {DESIRED_NEW_TOKENS}!)")
text_buggy = tokenizer.decode(out_buggy[0, diag_prompt_len:], skip_special_tokens=True)
print(f"text (truncated): {text_buggy!r}")

NotImplementedError: TODO: format_prompt

**Task:**
1. Run the code with a short prompt and observe: how many **new** tokens
   were actually produced, relative to how many the code's author
   **wanted** (the `DESIRED_NEW_TOKENS` variable)?
2. Diagnose the cause: what exactly does `max_length` count, and what
   does `max_new_tokens` count? (See the lecture, "The context window
   and truncation" section.)
3. Run the **same** code with a **longer** prompt (long enough that
   `prompt_len >= max_length`) -- what happens now? (Hint: different,
   louder behavior than with the short prompt.)
4. Fix the code (`max_new_tokens=` instead of `max_length=`).
5. Verify the fix: for **both** prompt lengths, the number of newly
   generated tokens must match what was intended.

In [15]:
# TODO:
# 1. as a comment: diagnose exactly what max_length counts here, and what we wanted
# 2. try the SAME buggy configuration with a LONGER prompt (prompt_len >= max_length)
#    - what happens now? (hint: a different, louder error)
# 3. fix: use max_new_tokens= instead of max_length=
# 4. verify the FIX for BOTH prompt lengths (assert on the number of new tokens)
raise NotImplementedError("TODO: diagnose and fix max_length/max_new_tokens")

NotImplementedError: TODO: diagnose and fix max_length/max_new_tokens

## 11. Conclusions -- your task

**1. Which decoding strategy gave the most useful results for the unambiguous-answer tasks (extraction, format)? Did sampling (where you tested it) help, hurt, or not matter?**

_(Your answer here.)_

**2. For the qualitative task -- did a higher temperature give subjectively better results? Did diversity matter for this specific task?**

_(Your answer here.)_

**3. What is the measured cost of your pipeline? How does it change with prompt length and with `use_cache`?**

_(Your answer here.)_

**4. What specific limitations of the model/pipeline did you observe?**

_(Your answer here.)_

**5. What is your conclusion about the quality/cost/configuration trade-off -- does a 'better' configuration always mean a better result for every task type?**

_(Your answer here.)_

---

## Extensions (optional, if time remains after sections 0-11)

- **R1.** Add 2 more prompts (e.g. a question with context, a text
  continuation) to the list from section 2, and give them the same
  treatment (sections 5, 6, 9) as the required prompts.
- **R2.** Also run sampling (both temperatures) for `format` (and for the
  R1 prompts, if you added them) -- does sampling help, hurt, or not
  matter for unambiguous tasks the core did not give it to?
- Add a **sixth** prompt of your own design, in a different style from
  the suggestions above.
- Measure the effect of `repetition_penalty` on text continuation (does
  it reduce phrase repetition for longer generation?).

_(This extension is optional -- code and results are not required in the core.)_